In [ ]:
import os
import numpy as np
from src.data_processor import DataProcessor
from src.model_trainer import ModelTrainer
from src.trajectory_predictor import TrajectoryPredictor
import matplotlib.pyplot as plt
import random
import tensorflow as tf
from pathlib import Path

SEED = 260330
BASE_DIR = os.getcwd()
WINDOW_SIZE = 200

TEST_DATA_CONFIGS = {
    "test_looking_3loop_l01.csv": {"skiprows": 50, "skipfooter": 50, "rotation_flag": False, "title": "S22U Looking Left 3-loop"},
    "test_looking_3loop_r03.csv": {"skiprows": 0, "skipfooter": 0, "rotation_flag": True, "title": "S22U Looking Right 3-loop"},
    "test_swing_3loop_l02.csv": {"skiprows": 50, "skipfooter": 50, "rotation_flag": False, "title": "S22U Swing Left 3-loop"},
    "test_swing_3loop_r01.csv": {"skiprows": 50, "skipfooter": 50, "rotation_flag": True, "title": "S22U Swing Right 3-loop"},
    "test_calling_3loop_l02.csv": {"skiprows": 0, "skipfooter": 50, "rotation_flag": False, "title": "S22U Calling Left 3-loop"},
    "test_calling_3loop_r04.csv": {"skiprows": 0, "skipfooter": 50, "rotation_flag": True, "title": "S22U Calling Right 3-loop"},
    
    "test_looking_3loop_l01_S20P.csv": {"skiprows": 50, "skipfooter": 50, "rotation_flag": False, "title": "S20+ Looking Left 3-loop"},
    "test_looking_3loop_r01_S20P.csv": {"skiprows": 0, "skipfooter": 0, "rotation_flag": True, "title": "S20+ Looking Right 3-loop"},
    "test_swing_3loop_l01_S20P.csv": {"skiprows": 50, "skipfooter": 50, "rotation_flag": False, "title": "S20+ Swing Left 3-loop"},
    "test_swing_3loop_r01_S20P.csv": {"skiprows": 50, "skipfooter": 50, "rotation_flag": True, "title": "S20+ Swing Right 3-loop"},
    "test_calling_3loop_l01_S20P.csv": {"skiprows": 100, "skipfooter": 50, "rotation_flag": False, "title": "S20+ Calling Left 3-loop"},
    "test_calling_3loop_r01_S20P.csv": {"skiprows": 150, "skipfooter": 150, "rotation_flag": True, "title": "S20+ Calling Right 3-loop"},
    
    "test3_looking_S22U.csv": {"skiprows": 50, "skipfooter": 50, "rotation_flag": False, "title": "S22U Looking figure-eight"},
    "test3_swing_S22U.csv": {"skiprows": 50, "skipfooter": 50, "rotation_flag": False, "title": "S22U Swing figure-eight"},
    "test3_calling_S22U.csv": {"skiprows": 120, "skipfooter": 50, "rotation_flag": False, "title": "S22U Calling figure-eight"},
    
    "test3_looking_S20P.csv": {"skiprows": 50, "skipfooter": 50, "rotation_flag": False, "title": "S20+ Looking figure-eight"},
    "test3_swing_S20P.csv": {"skiprows": 50, "skipfooter": 50, "rotation_flag": False, "title": "S20+ Swing figure-eight"},
    "test3_calling_S20P.csv": {"skiprows": 220, "skipfooter": 50, "rotation_flag": False, "title": "S20+ Calling figure-eight"},
}


TEST_DATA_PATHS = [
    # ============================================================
    # TEST BED 1: 대학본관 3층 3회 루프 데이터
    # ============================================================
    
    # S22U 테스트 데이터 
    os.path.join(BASE_DIR, "data", "tester1", "test_data", "looking", "test_looking_3loop_l01.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data", "looking", "test_looking_3loop_r03.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data", "swing", "test_swing_3loop_l02.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data", "swing", "test_swing_3loop_r01.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data", "calling", "test_calling_3loop_l02.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data", "calling", "test_calling_3loop_r04.csv"),

    # S20+ 테스트 데이터 
    os.path.join(BASE_DIR, "data", "tester1", "test_data", "looking", "test_looking_3loop_l01_S20P.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data", "looking", "test_looking_3loop_r01_S20P.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data", "swing", "test_swing_3loop_l01_S20P.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data", "swing", "test_swing_3loop_r01_S20P.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data", "calling", "test_calling_3loop_l01_S20P.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data", "calling", "test_calling_3loop_r01_S20P.csv"),
    
    # # ============================================================
    # # TEST BED 2: 대학본관 3층 1회 8자 루프 데이터 
    # # ============================================================
    # # S22U 테스트 데이터 
    os.path.join(BASE_DIR, "data", "tester1", "test_data3", "test3_looking_S22U.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data3", "test3_swing_S22U.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data3", "test3_calling_S22U.csv"),

    # S20+ 테스트 데이터 
    os.path.join(BASE_DIR, "data", "tester1", "test_data3", "test3_looking_S20P.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data3", "test3_swing_S20P.csv"),
    os.path.join(BASE_DIR, "data", "tester1", "test_data3", "test3_calling_S20P.csv"),
]

In [ ]:
SEED = 260330
BASE_DIR = os.getcwd()
WINDOW_SIZE = 200

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

model_path = os.path.join(BASE_DIR, "saved_models", "LSTM64_32_b512.h5")

trainer = ModelTrainer(WINDOW_SIZE, num_features=7)
trainer.load_model(model_path)

predictor = TrajectoryPredictor(
    trainer.model,
    trainer.x_scaler,
    trainer.y_scaler,
    window_size=WINDOW_SIZE,
)

for test_path in TEST_DATA_PATHS:
    if not os.path.exists(test_path):
        print(f"테스트 파일을 찾을 수 없습니다: {test_path}")
        continue

    file_name = Path(test_path).name
    cfg = TEST_DATA_CONFIGS.get(file_name, {})

    skiprows = cfg.get("skiprows", 0)
    rotation_flag = cfg.get("rotation_flag", False)
    title = cfg.get("title", file_name)
    skipfooter = cfg.get("skipfooter", 50)

    print(f"테스트 파일 로드 중: {title}")

    df_test = DataProcessor.load_and_preprocess_csv_test(
        test_path,
        skiprows=skiprows,
        skipfooter=skipfooter,
    )

    pred_X, pred_Y = predictor.predict_and_plot_trajectory(
        df_test,
        rotation_flag=rotation_flag,
    )